# Léia: ajuste fino do encoder da busca

Ajusta `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` com pedidos sintéticos de
leitores em português e mede o ganho num conjunto de **livros que o treino nunca viu**.
Código, desenho da avaliação e resultados: [github.com/nathanjrdev/leia](https://github.com/nathanjrdev/leia)
(`evals/embeddings/`).

**Antes de rodar:** em *Settings*, ligue **Accelerator: GPU** e **Internet**. Anexe (*Add Input*)
o dataset com `pares_treino.jsonl` e `pares_validacao.jsonl`, gerados por `gerar_pares.py`.
Para publicar no Hugging Face, crie o segredo `HF_TOKEN` em *Add-ons → Secrets*.

In [ ]:
BRANCH = "feat/embeddings-finetune"
EPOCAS = 3
PUBLICAR = False   # True envia o modelo e o dataset ao Hugging Face (privados)

In [ ]:
!git clone --depth 1 -b {BRANCH} https://github.com/nathanjrdev/leia.git
!pip install -q -U "sentence-transformers>=5" datasets accelerate huggingface_hub python-dotenv

## Pares de treino e validação
Vêm do dataset anexado. A divisão é por livro e já foi feita por `gerar_pares.py`.

In [ ]:
import glob, json, shutil
from pathlib import Path

saida = Path("leia/evals/embeddings/saida")
saida.mkdir(parents=True, exist_ok=True)
for nome in ("pares_treino.jsonl", "pares_validacao.jsonl"):
    achados = glob.glob(f"/kaggle/input/**/{nome}", recursive=True)
    assert achados, f"{nome} não encontrado: anexe o dataset com os pares"
    shutil.copy(achados[0], saida / nome)

for nome in ("pares_treino.jsonl", "pares_validacao.jsonl"):
    linhas = [json.loads(l) for l in open(saida / nome, encoding="utf-8") if l.strip()]
    livros = {l["livro"] for l in linhas}
    print(f"{nome}: {len(linhas)} pares, {len(livros)} livros")
print("\nAmostra:")
for l in linhas[:5]:
    print(" -", l["consulta"], "→", l["livro"])

## Treino
`MultipleNegativesRankingLoss`, lotes sem dois pedidos do mesmo livro.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (ligue em Settings)")
!cd leia && python evals/embeddings/treinar.py --epocas {EPOCAS}

## Avaliação: antes e depois
Busca só por vetor sobre o catálogo inteiro. O número principal é `validacao_livros_nao_vistos`.

In [ ]:
!cd leia && python evals/embeddings/avaliar.py

## Arquivos para baixar
O JSON de resultados vai para `evals/resultados/` no repositório; o modelo, se for usar localmente.

In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working")
for arquivo in Path("leia/evals/resultados").glob("embeddings_ajuste_*.json"):
    shutil.copy(arquivo, out / arquivo.name)
shutil.make_archive(str(out / "leia_encoder_ajustado"), "zip", "leia/evals/embeddings/saida/modelo")
print(sorted(p.name for p in out.iterdir()))

## Publicação no Hugging Face (opcional)
Monta o model card com os números da avaliação acima e envia como **privado**. Revise no site
antes de tornar público. O token vem do segredo `HF_TOKEN` e nunca é impresso.

In [ ]:
import os
!cd leia && python evals/embeddings/publicar_hf.py   # simulação: só monta as pastas
if PUBLICAR:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    !cd leia && python evals/embeddings/publicar_hf.py --confirmar